# 04-4 처음 보는 음식: CLIP만 vs CLIP + LLM 교차 확인 (+ 앱 기준값 0.8 vs 0.95)
**왜:** 발표에서 "LLM은 판정하지 않고 두 번째 의견만 낸다"는 역할의 **근거 숫자**가 필요하다. 06에서는 카오소이 한 장을 CLIP이 세비체로 찾아 밀을 놓쳤다.
또 기준값 `T_ONE`을 한솔님 제안 0.8로 할지 0.95로 할지도 같은 사진으로 비교한다.

| 비교 | 내용 |
|---|---|
| A. 2단계만 | 모든 사진을 2단계로: **CLIP만** vs **CLIP + LLM 교차 확인** (`search.crosscheck`, 앱과 같은 코드) |
| B. 앱 전체 | 분류 모델(Base, 보정) → `judge.analyze` 그대로. `T_ONE` 0.8 / 0.95 × LLM 끔 / 켬 |

- 사진: 03에서 모은 레시피 사진. **처음 보는 음식** = 분류 모델이 모르는 74종 (음식마다 `N_NEW`장), **아는 음식** = Food-101 101종 (음식마다 `N_KNOWN`장, 괜한 경고 확인용)
- 정답: 그 음식의 표(`food_table.csv`) 비율 → 주의 / 확인 필요 / 해당 없음. 레시피 30개 미만 음식은 정답이 불확실해서 뺀다
- **반칙 방지:** 시험 사진이 나온 레시피 글은 CLIP 검색 목록에서 뺀다 (자기 글 찾기 방지, 04와 같은 규칙)
- 지표 (사진 × 19항목): `주의놓침` = 정답이 주의인데 `해당 없음` (낮을수록 좋음, 04와 같은 정의) / `괜한 경고` = 정답이 해당 없음인데 주의·확인 필요 / `단계일치`
- LLM: 왼쪽 🔑 보안 비밀에 키 (06과 같음, 키 담당 서준영). LLM 답은 드라이브 `찍먹_실험기록/llm_cache_04-4/`에 저장 → 끊겨도 다시 돌리면 이어서 함
- 런타임: **CPU로 충분** (앱과 같은 CPU 경로). 사진 준비 약 15분 + LLM (무료 한도 때문에 호출 사이 `LLM_WAIT`초 쉼)
- 결과: 드라이브 `찍먹_실험기록/표/04-4_*.csv`

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
print('준비 완료 (GPU 없어도 됨: 서버는 CPU로 돌림)', torch.cuda.is_available())

## 1. 앱 파일 복사 + 설치 (06과 같음)

In [ ]:
import shutil
W, IDX = f'{REPO}/modeling/weights', f'{REPO}/knowledge/index'
os.makedirs(W, exist_ok=True); os.makedirs(IDX, exist_ok=True)
MODEL = 'convnext_base_exp004'   # 한솔 최종 모델 exp004 (test 93.46%, 12에폭 최고). 예전: 'convnext_base'(exp003) / 'convnext_tiny'
shutil.copy(f'{EXP}/best/{MODEL}/best.pt', f'{W}/best.pt')
for f in ('index_emb.npy', 'index_meta.parquet'):
    shutil.copy(f'{EXP}/best/clip_base/{f}', f'{IDX}/{f}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
print('복사·설치 완료:', sorted(os.listdir(W)), sorted(os.listdir(IDX)))

## 2. LLM 키 — 보안 비밀에서 읽기 (06과 같음)

In [ ]:
from google.colab import userdata
for k in ('GEMINI_API_KEY', 'GROQ_API_KEY', 'LLM_PROVIDER', 'GEMINI_MODEL', 'GROQ_MODEL'):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v          # 값은 출력하지 않음
    except Exception:                  # 없음 / 노트북 액세스 꺼짐
        pass
print('LLM 제공자:', os.environ.get('LLM_PROVIDER', 'gemini'),
      '/ 키 있음:', any(k in os.environ for k in ('GEMINI_API_KEY', 'GROQ_API_KEY')))

## 3. 설정

In [ ]:
N_NEW, N_KNOWN = 5, 2   # 음식당 사진 수 (처음 보는 74종 / 아는 101종). LLM 무료 한도가 넉넉하면 늘리기
LLM_WAIT = 4            # 초. LLM을 실제로 부를 때만 쉼 (저장된 답은 바로)

## 4. 사진 고르기 → 분류 후보 + CLIP 검색 (자기 레시피 글 제외)

In [ ]:
import pickle
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from PIL import Image
from knowledge import search
from knowledge.lookup import ALLERGENS, row as table_row
from modeling.model import load_model, predict

OUT = f'{EXP}/표'
CACHE = f'{EXP}/표/04-4_사진기록.pkl'   # 분류 후보 + CLIP 결과 (다시 돌리면 이어서)
classes = open(f'{REPO}/modeling/classes.txt').read().split()
new_foods = pd.read_csv(f'{REPO}/knowledge/candidates.csv', encoding='utf-8-sig')['class'].tolist()
table = pd.read_csv(f'{REPO}/knowledge/food_table.csv', encoding='utf-8-sig').set_index('class')
fake = set(pd.read_csv(f'{IMG}/_fake_md5.csv').md5)
ok = lambda c: c in table.index and table.loc[c, 'n_recipes'] >= 30   # 정답이 믿을 만한 음식만

def pick(label, n):
    p = f'{IMG}/{label}.parquet'
    if not os.path.exists(p):
        return []
    d = pd.read_parquet(p, columns=['recipe_idx', 'jpg', 'md5'])
    d = d[~d.md5.isin(fake)].drop_duplicates('md5')
    return [dict(label=label, recipe_idx=int(r.recipe_idx), md5=r.md5, jpg=r.jpg)
            for r in d.sample(min(n, len(d)), random_state=0).itertuples()]

photos = [dict(p, group='처음 보는 음식') for c in new_foods if ok(c) for p in pick(c, N_NEW)] + \
         [dict(p, group='아는 음식') for c in classes if ok(c) for p in pick(c, N_KNOWN)]

cnx = load_model()                        # 앱과 같은 모델·보정 (modeling/weights/best.pt + calibration.json)
clip, tf, emb, meta = search._load()
print('분류 모델:', cnx[0].kind, '/ 보정 T =', cnx[0].temperature)

rec = pickle.load(open(CACHE, 'rb')) if os.path.exists(CACHE) else {}
t0 = time.time()
for i, p in enumerate(photos):
    key = (p['label'], p['md5'])
    if key in rec:
        continue
    try:
        img = Image.open(io.BytesIO(p['jpg'])).convert('RGB')
    except Exception:
        continue
    with torch.no_grad():
        f = F.normalize(clip.visual_projection(clip.vision_model(pixel_values=tf(img)[None]).pooler_output), dim=-1)[0]
    sim = emb @ f
    sim[torch.from_numpy(((meta.recipe_idx == p['recipe_idx']) & (meta.label == p['label'])).values)] = -1e9  # 자기 레시피 글 빼기
    s, idx = sim.topk(search.K)
    hit = meta.iloc[idx.numpy()]
    rec[key] = dict(group=p['group'], label=p['label'], cands=predict(cnx, img),
                    clip=search.aggregate(s.numpy(), hit.label.values, hit[ALLERGENS].values))
    if len(rec) % 50 == 0:
        pickle.dump(rec, open(CACHE, 'wb')); print(f'{i + 1}/{len(photos)}  {time.time() - t0:.0f}초')
pickle.dump(rec, open(CACHE, 'wb'))
imgs = {(p['label'], p['md5']): p['jpg'] for p in photos}
rec = {k: v for k, v in rec.items() if k in imgs}
print(pd.Series([r['group'] for r in rec.values()]).value_counts().to_dict(),
      '| 처음 보는 음식', len({r['label'] for r in rec.values() if r['group'] == '처음 보는 음식'}), '종')

## 5. LLM 이름 추정

In [ ]:
# LLM 이름·재료 추정 (앱과 같은 llm.guess). 답은 드라이브에 저장 → 다시 돌리면 이어서
import llm.client as lc
from pathlib import Path
from llm.guess import guess_name, guess_ingredients
lc.CACHE = Path(f'{EXP}/llm_cache_04-4')      # 앱의 llm/_cache 대신 드라이브 (런타임이 끊겨도 남음)
LLM_ON = any(k in os.environ for k in ('GEMINI_API_KEY', 'GROQ_API_KEY'))
fail = {'name': 0, 'ing': 0}

def _call(fn, arg, kind):
    n = len(list(lc.CACHE.glob('*.json'))) if lc.CACHE.exists() else 0
    out = fn(arg)
    if lc.CACHE.exists() and len(list(lc.CACHE.glob('*.json'))) > n:
        time.sleep(LLM_WAIT)                   # 새로 물어봤을 때만 쉼
    if out is None:
        fail[kind] += 1
    return out

names, ings = {}, {}
def llm_name(key):
    if key not in names:
        names[key] = _call(guess_name, Image.open(io.BytesIO(imgs[key])).convert('RGB'), 'name') if LLM_ON else None
    return names[key]
def llm_ing(name):
    if name not in ings:
        ings[name] = _call(guess_ingredients, name, 'ing') if LLM_ON else None
    return ings[name]

print('LLM 키 있음:', LLM_ON, '| 제공자:', lc.provider(), lc.model_name())
if LLM_ON:   # 처음 보는 음식 사진은 전부 필요 (A 비교). 아는 음식은 앱이 2단계·합치기로 보낼 때만 아래에서 부름
    keys_new = [k for k, r in rec.items() if r['group'] == '처음 보는 음식']
    for i, k in enumerate(keys_new):
        llm_name(k)
        if (i + 1) % 25 == 0:
            print(f'{i + 1}/{len(keys_new)}  실패 {fail}', '| 마지막 오류:', lc.last_error)
    print('이름 추정 실패:', fail, '| 마지막 오류:', lc.last_error)
    print('→ 실패가 많으면(하루 한도 등) 결과가 "CLIP만"과 같아짐. 내일 이 셀부터 다시 돌리면 저장된 답은 건너뜀')

## 6. 비교: 주의놓침 · 괜한 경고 · 경로

In [ ]:
import knowledge.judge as J

def crossed(key):
    r = rec[key]
    return search.crosscheck(r['clip'], llm_name(key), llm_ing)

def score(verdicts, label):
    """verdicts: {알레르기: 단계} 19개 → 사진 한 장의 [(정답 단계, 예측 단계)]"""
    return [(J.level(float(table.loc[label, a])), verdicts[a]) for a in ALLERGENS]

def metrics(pairs):
    t, p = np.array([x for x, _ in pairs]), np.array([y for _, y in pairs])
    return {'주의놓침 ↓': (p[t == '주의'] == '해당 없음').mean(),
            '괜한 경고 ↓': np.isin(p[t == '해당 없음'], ['주의', '확인 필요']).mean(),
            '단계일치 ↑': (t == p).mean(), '정답 주의 수': int((t == '주의').sum())}

ARMS = {'2단계만: CLIP만': ('stage2', False), '2단계만: CLIP + LLM': ('stage2', True)}
for t_one in (0.8, 0.95):
    for on in (False, True):
        ARMS[f'앱 전체: T_ONE {t_one}, LLM {"켬" if on else "끔"}'] = (t_one, on)

rows, routes, name_acc = [], [], []
for arm, (mode, use_llm) in ARMS.items():
    if use_llm and not LLM_ON:
        continue
    for group in ('처음 보는 음식', '아는 음식'):
        pairs, rt = [], []
        for key, r in rec.items():
            if r['group'] != group:
                continue
            stage2 = (lambda k=key: crossed(k)) if use_llm else (lambda r=r: r['clip'])
            if mode == 'stage2':
                v = J.judge(stage2()[0], ALLERGENS)['verdict']
            else:
                J.T_ONE, J.T_KNOWN = mode, 0.6
                out = J.analyze(r['cands'], ALLERGENS, stage2, lambda r=r: r['clip'])   # 앱처럼 CLIP을 먼저 봄 (소바→라멘 착각 확인)
                v = out['verdict']; rt.append(out['route'])
            pairs += score(v, r['label'])
        rows.append({'방법': arm, '사진': group, '장수': len(pairs) // len(ALLERGENS), **metrics(pairs)})
        if rt:
            routes.append({'방법': arm, '사진': group, **pd.Series(rt).value_counts(normalize=True).round(3).to_dict()})
J.T_ONE, J.T_KNOWN = 0.95, 0.6   # 앱 값으로 되돌림

res = pd.DataFrame(rows).round(3)
res.to_csv(f'{OUT}/04-4_주의놓침.csv', index=False, encoding='utf-8-sig')
display(res)
res_route = pd.DataFrame(routes).fillna(0)
res_route.to_csv(f'{OUT}/04-4_경로.csv', index=False, encoding='utf-8-sig')
display(res_route)

## 7. 이름 맞히기

In [ ]:
# 이름 맞히기 (처음 보는 음식): CLIP 검색 1등·상위 3개 vs LLM 1등. 교차 확인이 언제 도움이 됐는지
rows = []
for key, r in rec.items():
    if r['group'] != '처음 보는 음식':
        continue
    top = [c for c, _ in r['clip'][1][:3]]
    g = names.get(key)
    rows.append({'음식': r['label'], 'CLIP 1등': top[0] == r['label'], 'CLIP 상위3': r['label'] in top,
                 'LLM 1등': bool(g) and g[0][0] == r['label'], 'LLM 답 있음': bool(g)})
nm = pd.DataFrame(rows)
summary = nm.drop(columns='음식').mean().round(3)
summary.to_frame('비율').to_csv(f'{OUT}/04-4_이름맞히기.csv', encoding='utf-8-sig')
display(summary.to_frame('비율'))
print('CLIP은 틀리고 LLM은 맞힌 음식 (교차 확인이 도움 된 곳):')
print(nm[~nm['CLIP 상위3'] & nm['LLM 1등']].음식.value_counts().head(15).to_dict())

## 끝 — 읽는 법
- **A (2단계만):** `처음 보는 음식` 줄에서 `CLIP + LLM`의 `주의놓침`이 `CLIP만`보다 낮으면 → LLM 교차 확인이 놓침을 줄인다는 근거. 대신 `괜한 경고`가 얼마나 늘었는지 같이 본다
  (LLM 근거는 `확인 필요`까지만 올리므로 `주의`가 새로 생기지는 않는다)
- **B (앱 전체):** `T_ONE 0.8` vs `0.95`에서 처음 보는 음식의 주의놓침과 아는 음식의 괜한 경고를 비교 → 한솔님과 기준값 합의 자료
- `04-4_경로.csv`: 처음 보는 음식이 `1단계-1등`(CLIP 확인 없이 틀린 표로 판정)으로 간 비율
- LLM 실패가 많았으면 숫자가 `CLIP만`과 비슷하게 나온다 → LLM 셀 출력의 실패 수부터 확인
- 다 돌리면 Claude에게 "04-4 다 돌렸어" → 결과를 `docs/진행상황.md`에 기록